> Adapted from *Dive into Deep Learning*, section [Parameter Initialization](https://d2l.ai/chapter_builders-guide/init-param.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 6.2: Parameter Management](06.02_parameters.ipynb) | Next: [Section 6.4: Lazy Initialization](06.04_lazy-init.ipynb)

# Parameter Initialization

Now that we know how to access the parameters,
let's look at how to initialize them properly.
We discussed the need for proper initialization in [Section 5.4](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html#sec-numerical-stability).
The deep learning framework provides default random initializations to its layers.
However, we often want to initialize our weights
according to various other protocols. The framework provides most commonly
used protocols, and also allows to create a custom initializer.

In [ ]:
import numpy as np
from divelet_ml import nn

import divelet_ml as d2l

By default, `divelet_ml` initializes, as PyTorch does, weight and bias matrices
uniformly by drawing from a range that is computed according to the input and output dimension.
`divelet_ml`'s `nn.init` provides a variety
of preset initialization methods.

In [ ]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
X = np.random.rand(2, 4)
net(X).shape

## Built-in Initialization

Let's begin by calling on built-in initializers.
The code below initializes all weight parameters
as Gaussian random variables
with standard deviation 0.01, while bias parameters are cleared to zero.

In [ ]:
def init_normal(module):
    if type(module) == nn.Linear:
        nn.init.normal_(module.weight, mean=0, std=0.01)
        nn.init.zeros_(module.bias)

net.apply(init_normal)
net[0].weight[0], net[0].bias[0]

We can also initialize all the parameters
to a given constant value (say, 1).

In [ ]:
def init_constant(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 1)
        nn.init.zeros_(module.bias)

net.apply(init_constant)
net[0].weight[0], net[0].bias[0]

We can also apply different initializers for certain blocks.
For example, below we initialize the first layer
with the Xavier initializer
and initialize the second layer
to a constant value of 42.

In [ ]:
def init_xavier(module):
    if type(module) == nn.Linear:
        nn.init.xavier_uniform_(module.weight)

def init_42(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight[0])
print(net[2].weight)

### Custom Initialization

Sometimes, the initialization methods we need
are not provided by the deep learning framework.
In the example below, we define an initializer
for any weight parameter $w$ using the following strange distribution:

$$
\begin{aligned}
    w \sim \begin{cases}
        U(5, 10) & \textrm{ with probability } \frac{1}{4} \\
            0    & \textrm{ with probability } \frac{1}{2} \\
        U(-10, -5) & \textrm{ with probability } \frac{1}{4}
    \end{cases}
\end{aligned}
$$


Again, we implement a `my_init` function to apply to `net`.

In [ ]:
def my_init(module):
    if type(module) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in module.named_parameters()][0])
        nn.init.uniform_(module.weight, -10, 10)
        module.weight *= np.abs(module.weight) >= 5

net.apply(my_init)
net[0].weight[:2]

Note that we always have the option
of setting parameters directly.

In [ ]:
net[0].weight[:] += 1
net[0].weight[0, 0] = 42
net[0].weight[0]

## Summary

We can initialize parameters using built-in and custom initializers.

## Exercises

Look up the online documentation for more built-in initializers.



Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/8090)

Previous: [Section 6.2: Parameter Management](06.02_parameters.ipynb) | Next: [Section 6.4: Lazy Initialization](06.04_lazy-init.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Parameter Initialization](https://d2l.ai/chapter_builders-guide/init-param.html)
(source: [`chapter_builders-guide/init-param.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_builders-guide/init-param.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`.
- `nn.init.normal_`, `uniform_`, `constant_`, `zeros_` and `xavier_uniform_` fill an array in place, as PyTorch's do. A lazy layer becomes an `nn.Linear` when its first input has given it its parameters, as in PyTorch, which is what `type(module) == nn.Linear` tests.
- `weight.data` is `weight`: the parameters are plain arrays, which the initializers and the cell that sets entries directly change in place.
- `torch.rand` is `np.random.rand`; `module.weight.data.abs()` is `np.abs(module.weight)`.
- Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.